# Notebook 02 — Build the 120-Image Manifest

**Objective.** Build and inspect the candidate 120-image benchmark manifest (six buckets).

**Inputs.** GQA scene graphs.

**Expected outputs.** Candidate manifest CSV, bucket distribution, review checklist.

_Companion notebook: runnable top-to-bottom, relative paths, no hidden state, saves artifacts to `results/`._

In [ ]:
import sys, os
# Reusable logic lives in src/grounded_memory/. Notebooks call it.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from pathlib import Path
RESULTS = Path('..') / 'results'
from grounded_memory import __version__
print('grounded_memory', __version__)

## Build candidate manifest

Score images per bucket and greedily fill six 20-image buckets.

In [ ]:
from grounded_memory.data.gqa import scene_graph_path, load_scene_graphs
from grounded_memory.data.selection import build_candidate_manifest, BUCKETS
try:
    sgs = load_scene_graphs(scene_graph_path(split='val'))
    rows = build_candidate_manifest(sgs, split='val', per_bucket=20)
    from collections import Counter
    print(Counter(r.primary_bucket for r in rows))
except FileNotFoundError as e:
    print('Need GQA first:', e)

## Manual review (REQUIRED before freezing)

Inspect each image, set `manually_reviewed=True`, fix ambiguity notes/buckets. Do not run experiments on an unreviewed manifest.

## Findings

_Fill in 3–5 concrete findings after running this notebook._

1. …
2. …
3. …

> Do not populate tables with invented numbers. If the full model/data is unavailable, leave clearly-labeled placeholders.

## Manual image-by-image preview

Steps through `data/manifests/gqa_benchmark_v1.csv` one row at a time: shows the
image and prints every column for that row. Re-run the cell below (Shift+Enter)
to advance to the next image — no need to edit any code between runs.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
MANIFEST_CSV = REPO_ROOT / 'data' / 'manifests' / 'gqa_benchmark_v1.csv'

# GQA_ROOT (e.g. from .env) is meant to be resolved relative to the repo root,
# not the notebook kernel's cwd -- resolve it explicitly so it works either way.
gqa_root = Path(os.environ.get('GQA_ROOT', 'data/raw/gqa'))
if not gqa_root.is_absolute():
    gqa_root = REPO_ROOT / gqa_root
GQA_IMAGES_DIR = gqa_root / 'images'

manifest_df = pd.read_csv(MANIFEST_CSV)
print(f'{len(manifest_df)} rows loaded from {MANIFEST_CSV}')
print(f'Images dir: {GQA_IMAGES_DIR} (exists: {GQA_IMAGES_DIR.exists()})')

In [ ]:
# Re-run this cell to step forward one image at a time (wraps around at the end).
if '_preview_idx' not in globals():
    _preview_idx = 0

row = manifest_df.iloc[_preview_idx]
image_path = GQA_IMAGES_DIR / f"{row['image_id']}.jpg"

print(f'--- row {_preview_idx} / {len(manifest_df) - 1} ---')
for col, val in row.items():
    print(f'{col}: {val}')

if image_path.exists():
    img = Image.open(image_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"image_id={row['image_id']}")
    plt.show()
else:
    print(f'Image not found: {image_path}')

_preview_idx = (_preview_idx + 1) % len(manifest_df)

## Manual review loop (sets `manually_reviewed` and saves)

Walks every row in order, shows the image + all columns, then asks whether to
mark it reviewed. Press **Enter** for `True`, type **n** + Enter to leave it
`False`. The CSV is saved after every row, so interrupting the kernel partway
through does not lose progress already made.

In [ ]:
for idx, row in manifest_df.iterrows():
    image_path = GQA_IMAGES_DIR / f"{row['image_id']}.jpg"

    print(f'--- row {idx} / {len(manifest_df) - 1} ---')
    for col, val in row.items():
        print(f'{col}: {val}')

    if image_path.exists():
        img = Image.open(image_path)
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"image_id={row['image_id']}")
        plt.show()
    else:
        print(f'Image not found: {image_path}')

    try:
        answer = input('Set manually_reviewed=True? [Enter=yes, n=no]: ').strip().lower()
    except KeyboardInterrupt:
        print(f'\nStopped at row {idx}. Progress so far is already saved to {MANIFEST_CSV}.')
        break

    manifest_df.at[idx, 'manually_reviewed'] = answer != 'n'
    manifest_df.to_csv(MANIFEST_CSV, index=False)

print(f'Done. {manifest_df["manually_reviewed"].sum()} / {len(manifest_df)} rows marked reviewed.')